In [1]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        has_root_files = (candidate / "train.csv").exists() and (candidate / "test.csv").exists()
        has_data_files = (candidate / "data" / "train.csv").exists() and (candidate / "data" / "test.csv").exists()
        if has_root_files or has_data_files:
            return candidate
    raise FileNotFoundError("Could not locate project root.")


PROJECT_ROOT = find_project_root()


def resolve_data_path(filename: str) -> Path:
    for path in [PROJECT_ROOT / filename, PROJECT_ROOT / "data" / filename]:
        if path.exists():
            return path
    raise FileNotFoundError(f"Could not find {filename} in project root or data/")


train = pd.read_csv(resolve_data_path("train.csv"), parse_dates=["date"])
test = pd.read_csv(resolve_data_path("test.csv"), parse_dates=["date"])

train_date_summary = (
    train.groupby("date")
    .agg(
        total_rows=("row_id", "size"),
        observed_rows=("iv_observed", lambda s: s.notna().sum()),
        missing_rows=("iv_observed", lambda s: s.isna().sum()),
        observed_ratio=("iv_observed", lambda s: s.notna().mean()),
    )
    .sort_index()
)

train_dates = train_date_summary.index.to_list()

N_FOLDS = 4
VAL_DATES_PER_FOLD = 5
TOTAL_VAL_DATES = N_FOLDS * VAL_DATES_PER_FOLD
val_start_idx = len(train_dates) - TOTAL_VAL_DATES

fold_plan = pd.DataFrame(
    [
        {
            "fold": fold_id + 1,
            "train_start": train_dates[0],
            "train_end": train_dates[val_start_idx + fold_id * VAL_DATES_PER_FOLD - 1],
            "n_train_dates": val_start_idx + fold_id * VAL_DATES_PER_FOLD,
            "val_start": train_dates[val_start_idx + fold_id * VAL_DATES_PER_FOLD],
            "val_end": train_dates[val_start_idx + (fold_id + 1) * VAL_DATES_PER_FOLD - 1],
            "n_val_dates": VAL_DATES_PER_FOLD,
        }
        for fold_id in range(N_FOLDS)
    ]
)

print("Project root:", PROJECT_ROOT)
print("Train shape:", train.shape, "| Test shape:", test.shape)
display(fold_plan)


Project root: /Users/sauravkrishna/Documents/projects/NQFO-Impilied-volatility-surface
Train shape: (11640, 10) | Test shape: (3960, 10)


,fold,train_start,train_end,n_train_dates,val_start,val_end,n_val_dates
0,1,2025-01-02,2025-04-18,77,2025-04-21,2025-04-25,5
1,2,2025-01-02,2025-04-25,82,2025-04-28,2025-05-02,5
2,3,2025-01-02,2025-05-02,87,2025-05-05,2025-05-09,5
3,4,2025-01-02,2025-05-09,92,2025-05-12,2025-05-16,5


In [2]:
BUCKET_COLS = ["maturity_label", "option_type"]
NODE_COLS = ["maturity_label", "moneyness", "option_type"]

MASK_PROTOCOL_CONFIG = {
    "stress_test": {
        "base_hide_rate": 0.10,
        "node_weight": 1.00,
        "support_weight": 0.00,
    },
    "primary_realistic": {
        "base_hide_rate": 0.10,
        "node_weight": 0.65,
        "support_weight": 0.35,
    },
}
LOCKED_MASK_SEED = "nqfo-val-v1"

OVERALL_TEST_MISSING_RATE = test["iv_observed"].isna().mean()

test_bucket_pattern = (
    test.assign(is_missing=test["iv_observed"].isna())
    .groupby(BUCKET_COLS)["is_missing"]
    .mean()
    .rename("test_bucket_missing_rate")
    .reset_index()
)

test_node_pattern = (
    test.assign(is_missing=test["iv_observed"].isna())
    .groupby(NODE_COLS)["is_missing"]
    .mean()
    .rename("test_node_missing_rate")
    .reset_index()
)

surface_levels = pd.concat(
    [
        train[["moneyness", "maturity_label", "maturity_days"]],
        test[["moneyness", "maturity_label", "maturity_days"]],
    ],
    ignore_index=True,
)
moneyness_levels = sorted(surface_levels["moneyness"].dropna().unique().tolist())
maturity_levels = (
    surface_levels[["maturity_label", "maturity_days"]]
    .drop_duplicates()
    .sort_values("maturity_days")["maturity_label"]
    .tolist()
)
m_idx = {m: i for i, m in enumerate(moneyness_levels)}
t_idx = {t: i for i, t in enumerate(maturity_levels)}


def stable_uniform(key: str) -> float:
    digest = hashlib.md5(key.encode("utf-8")).hexdigest()
    return int(digest[:12], 16) / float(16**12 - 1)


def opposite_option(opt: str) -> str:
    return "put" if opt == "call" else "call"


def local_support_profile(target_rows: pd.DataFrame, visible_rows: pd.DataFrame) -> pd.DataFrame:
    prof = target_rows.copy()
    visible_key_set = set(
        zip(
            visible_rows["date"],
            visible_rows["moneyness"],
            visible_rows["maturity_label"],
            visible_rows["option_type"],
        )
    )

    opp_visible = []
    same_maturity_adj_count = []
    same_moneyness_adj_count = []

    for d, m, t, o in zip(
        prof["date"],
        prof["moneyness"],
        prof["maturity_label"],
        prof["option_type"],
    ):
        opp_visible.append((d, m, t, opposite_option(o)) in visible_key_set)

        i = m_idx[m]
        j = t_idx[t]

        same_maturity_candidates = []
        if i - 1 >= 0:
            same_maturity_candidates.append((d, moneyness_levels[i - 1], t, o))
        if i + 1 < len(moneyness_levels):
            same_maturity_candidates.append((d, moneyness_levels[i + 1], t, o))

        same_moneyness_candidates = []
        if j - 1 >= 0:
            same_moneyness_candidates.append((d, m, maturity_levels[j - 1], o))
        if j + 1 < len(maturity_levels):
            same_moneyness_candidates.append((d, m, maturity_levels[j + 1], o))

        same_maturity_adj_count.append(sum(c in visible_key_set for c in same_maturity_candidates))
        same_moneyness_adj_count.append(sum(c in visible_key_set for c in same_moneyness_candidates))

    prof["opp_option_visible"] = opp_visible
    prof["same_maturity_adj_visible_count"] = same_maturity_adj_count
    prof["same_moneyness_adj_visible_count"] = same_moneyness_adj_count
    prof["local_support_score"] = (
        prof["opp_option_visible"].astype(int)
        + prof["same_maturity_adj_visible_count"]
        + prof["same_moneyness_adj_visible_count"]
    )
    return prof


def build_masked_validation_rows_with_protocol(
    full_df: pd.DataFrame,
    val_dates,
    protocol_name: str,
    seed: str = LOCKED_MASK_SEED,
) -> pd.DataFrame:
    cfg = MASK_PROTOCOL_CONFIG[protocol_name]

    val_df = full_df.loc[full_df["date"].isin(val_dates)].copy()
    val_df["is_orig_observed"] = val_df["iv_observed"].notna()
    val_df["is_orig_missing"] = ~val_df["is_orig_observed"]

    val_df = val_df.merge(test_bucket_pattern, on=BUCKET_COLS, how="left")
    val_df = val_df.merge(test_node_pattern, on=NODE_COLS, how="left")

    val_df["bucket_hide_rate_on_observed"] = (
        cfg["base_hide_rate"] * val_df["test_bucket_missing_rate"] / OVERALL_TEST_MISSING_RATE
    )

    val_df["priority_noise"] = val_df.apply(
        lambda r: stable_uniform(
            f"{seed}|{protocol_name}|{r['date'].date()}|{r['maturity_label']}|{r['moneyness']:.4f}|{r['option_type']}"
        ),
        axis=1,
    )

    observed_pool = val_df.loc[val_df["is_orig_observed"]].copy()
    observed_support = local_support_profile(observed_pool, observed_pool)[
        ["row_id", "local_support_score"]
    ]
    val_df = val_df.merge(observed_support, on="row_id", how="left")
    val_df["local_support_score"] = val_df["local_support_score"].fillna(0).astype(int)

    val_df["is_pseudo_hidden"] = False

    observed_pool = val_df.loc[val_df["is_orig_observed"]].copy()
    for _, g in observed_pool.groupby(["date", *BUCKET_COLS], sort=False):
        n_obs = len(g)
        n_hide = int(np.round(g["bucket_hide_rate_on_observed"].iloc[0] * n_obs))
        if n_hide <= 0:
            continue

        node_rank = g["test_node_missing_rate"].rank(method="average", pct=True)
        support_rank = g["local_support_score"].rank(method="average", pct=True)

        selection_priority = (
            cfg["node_weight"] * node_rank
            + cfg["support_weight"] * support_rank
            + 1e-6 * g["priority_noise"]
        )

        chosen_idx = (
            g.assign(selection_priority=selection_priority)
            .sort_values(["selection_priority", "row_id"], ascending=[False, True])
            .head(n_hide)
            .index
        )
        val_df.loc[chosen_idx, "is_pseudo_hidden"] = True

    val_df["is_scored_target"] = val_df["is_pseudo_hidden"]
    val_df["is_visible_anchor"] = val_df["is_orig_observed"] & ~val_df["is_pseudo_hidden"]
    val_df["is_effectively_missing"] = val_df["is_orig_missing"] | val_df["is_pseudo_hidden"]
    val_df["mask_protocol"] = protocol_name

    return val_df.sort_values(["date", "maturity_days", "option_type", "moneyness"]).reset_index(drop=True)


def make_fold_bundle_for_protocol(train_df: pd.DataFrame, fold_row, protocol_name: str):
    val_dates = train_date_summary.loc[fold_row.val_start:fold_row.val_end].index.tolist()
    train_dates = train_date_summary.loc[:fold_row.train_end].index.tolist()

    train_history = train_df.loc[train_df["date"].isin(train_dates)].copy()
    val_masked = build_masked_validation_rows_with_protocol(train_df, val_dates, protocol_name)

    return {
        "fold": fold_row.fold,
        "protocol": protocol_name,
        "train_history": train_history,
        "val_masked": val_masked,
        "val_visible_anchors": val_masked.loc[val_masked["is_visible_anchor"]].copy(),
        "val_scored_targets": val_masked.loc[val_masked["is_scored_target"]].copy(),
    }


def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float))))


def score_predictions(scored_df: pd.DataFrame, pred_col: str = "iv_pred") -> dict:
    eval_df = scored_df.loc[scored_df["is_scored_target"]].copy()
    eval_df = eval_df.loc[eval_df["iv_observed"].notna() & eval_df[pred_col].notna()].copy()
    return {
        "n_scored": len(eval_df),
        "rmse": rmse(eval_df["iv_observed"], eval_df[pred_col]),
    }


In [3]:
def predict_global_median(train_history: pd.DataFrame, val_masked: pd.DataFrame) -> pd.DataFrame:
    out = val_masked.copy()
    global_median = train_history.loc[train_history["iv_observed"].notna(), "iv_observed"].median()
    out["iv_pred"] = global_median
    return out


def predict_node_median(train_history: pd.DataFrame, val_masked: pd.DataFrame) -> pd.DataFrame:
    out = val_masked.copy()

    observed_train = train_history.loc[train_history["iv_observed"].notna()].copy()
    global_median = observed_train["iv_observed"].median()

    node_lookup = (
        observed_train.groupby(NODE_COLS)["iv_observed"]
        .median()
        .rename("node_median_pred")
        .reset_index()
    )

    out = out.merge(node_lookup, on=NODE_COLS, how="left")
    out["iv_pred"] = out["node_median_pred"].fillna(global_median)
    return out


def evaluate_baseline(protocol_name: str, predictor_fn, baseline_name: str) -> pd.DataFrame:
    bundles = [
        make_fold_bundle_for_protocol(train, row, protocol_name=protocol_name)
        for row in fold_plan.itertuples(index=False)
    ]

    rows = []
    for bundle in bundles:
        scored = predictor_fn(bundle["train_history"], bundle["val_masked"])
        metrics = score_predictions(scored, pred_col="iv_pred")
        rows.append(
            {
                "protocol": protocol_name,
                "baseline": baseline_name,
                "fold": bundle["fold"],
                "n_scored": metrics["n_scored"],
                "rmse": metrics["rmse"],
            }
        )
    return pd.DataFrame(rows)


In [4]:
results = pd.concat(
    [
        evaluate_baseline("primary_realistic", predict_global_median, "global_median"),
        evaluate_baseline("primary_realistic", predict_node_median, "node_median"),
        evaluate_baseline("stress_test", predict_global_median, "global_median"),
        evaluate_baseline("stress_test", predict_node_median, "node_median"),
    ],
    ignore_index=True,
)

display(results.sort_values(["protocol", "baseline", "fold"]))

summary = (
    results.groupby(["protocol", "baseline"])
    .agg(
        mean_rmse=("rmse", "mean"),
        min_rmse=("rmse", "min"),
        max_rmse=("rmse", "max"),
        mean_n_scored=("n_scored", "mean"),
    )
    .reset_index()
    .sort_values(["protocol", "mean_rmse"])
)

print("Baseline summary")
display(summary)


,protocol,baseline,fold,n_scored,rmse
0,primary_realistic,global_median,1,39,3.2421
1,primary_realistic,global_median,2,38,5.6385
2,primary_realistic,global_median,3,39,4.5625
3,primary_realistic,global_median,4,40,4.4262
4,primary_realistic,node_median,1,39,1.8828
5,primary_realistic,node_median,2,38,4.6171
6,primary_realistic,node_median,3,39,4.0958
7,primary_realistic,node_median,4,40,3.8880
8,stress_test,global_median,1,39,3.6372
9,stress_test,global_median,2,38,6.3833


Baseline summary


,protocol,baseline,mean_rmse,min_rmse,max_rmse,mean_n_scored
1,primary_realistic,node_median,3.6209,1.8828,4.6171,39.0000
0,primary_realistic,global_median,4.4673,3.2421,5.6385,39.0000
3,stress_test,node_median,3.6124,1.6997,4.7309,39.0000
2,stress_test,global_median,5.0479,3.6372,6.3833,39.0000


In [5]:
def build_node_lookup(observed_df: pd.DataFrame, pred_name: str) -> pd.DataFrame:
    return (
        observed_df.groupby(NODE_COLS)["iv_observed"]
        .median()
        .rename(pred_name)
        .reset_index()
    )


def predict_recent_node_median(
    train_history: pd.DataFrame,
    val_masked: pd.DataFrame,
    lookback_dates: int = 20,
) -> pd.DataFrame:
    out = val_masked.copy()

    observed_train = train_history.loc[train_history["iv_observed"].notna()].copy()
    global_median = observed_train["iv_observed"].median()

    recent_dates = sorted(observed_train["date"].unique())[-lookback_dates:]
    recent_obs = observed_train.loc[observed_train["date"].isin(recent_dates)].copy()

    recent_lookup = build_node_lookup(recent_obs, "recent_node_pred")
    full_lookup = build_node_lookup(observed_train, "full_node_pred")

    out = out.merge(recent_lookup, on=NODE_COLS, how="left")
    out = out.merge(full_lookup, on=NODE_COLS, how="left")

    out["iv_pred"] = (
        out["recent_node_pred"]
        .fillna(out["full_node_pred"])
        .fillna(global_median)
    )

    out["pred_source"] = np.select(
        [
            out["recent_node_pred"].notna(),
            out["full_node_pred"].notna(),
        ],
        [
            "recent_node_median",
            "full_node_median",
        ],
        default="global_median",
    )

    return out


In [7]:
def predict_same_date_linear_interp(
    train_history: pd.DataFrame,
    val_masked: pd.DataFrame,
    lookback_dates: int = 20,
) -> pd.DataFrame:
    out = predict_recent_node_median(train_history, val_masked, lookback_dates=lookback_dates).copy()

    out["interp_pred"] = np.nan
    out["interp_source"] = pd.Series(index=out.index, dtype="object")

    for (_, maturity_label, option_type), g_idx in out.groupby(
        ["date", "maturity_label", "option_type"], sort=False
    ).groups.items():
        g = out.loc[g_idx].copy()

        anchors = (
            g.loc[g["is_visible_anchor"], ["moneyness", "iv_observed"]]
            .dropna()
            .sort_values("moneyness")
        )

        if len(anchors) == 0:
            continue

        x = anchors["moneyness"].to_numpy()
        y = anchors["iv_observed"].to_numpy()

        if len(anchors) == 1:
            interp_vals = np.repeat(y[0], len(g))
            interp_label = "same_date_single_anchor"
        else:
            interp_vals = np.interp(g["moneyness"].to_numpy(), x, y)
            interp_label = "same_date_linear_interp"

        out.loc[g.index, "interp_pred"] = interp_vals
        out.loc[g.index, "interp_source"] = interp_label

    use_interp = out["interp_pred"].notna()
    out.loc[use_interp, "iv_pred"] = out.loc[use_interp, "interp_pred"]
    out["pred_source"] = np.where(use_interp, out["interp_source"], out["pred_source"])

    return out


In [8]:
new_results = pd.concat(
    [
        evaluate_baseline("primary_realistic", lambda th, vm: predict_recent_node_median(th, vm, lookback_dates=20), "recent_node_median_20d"),
        evaluate_baseline("primary_realistic", lambda th, vm: predict_same_date_linear_interp(th, vm, lookback_dates=20), "same_date_linear_interp"),
        evaluate_baseline("stress_test", lambda th, vm: predict_recent_node_median(th, vm, lookback_dates=20), "recent_node_median_20d"),
        evaluate_baseline("stress_test", lambda th, vm: predict_same_date_linear_interp(th, vm, lookback_dates=20), "same_date_linear_interp"),
    ],
    ignore_index=True,
)

results_phase3 = pd.concat([results, new_results], ignore_index=True)

display(results_phase3.sort_values(["protocol", "baseline", "fold"]))

summary_phase3 = (
    results_phase3.groupby(["protocol", "baseline"])
    .agg(
        mean_rmse=("rmse", "mean"),
        min_rmse=("rmse", "min"),
        max_rmse=("rmse", "max"),
        mean_n_scored=("n_scored", "mean"),
    )
    .reset_index()
    .sort_values(["protocol", "mean_rmse"])
)

print("Expanded baseline summary")
display(summary_phase3)


,protocol,baseline,fold,n_scored,rmse
0,primary_realistic,global_median,1,39,3.2421
1,primary_realistic,global_median,2,38,5.6385
2,primary_realistic,global_median,3,39,4.5625
3,primary_realistic,global_median,4,40,4.4262
4,primary_realistic,node_median,1,39,1.8828
5,primary_realistic,node_median,2,38,4.6171
6,primary_realistic,node_median,3,39,4.0958
7,primary_realistic,node_median,4,40,3.8880
16,primary_realistic,recent_node_median_20d,1,39,2.6329
17,primary_realistic,recent_node_median_20d,2,38,4.2711


Expanded baseline summary


,protocol,baseline,mean_rmse,min_rmse,max_rmse,mean_n_scored
3,primary_realistic,same_date_linear_interp,1.1877,0.9115,1.5942,39.0000
1,primary_realistic,node_median,3.6209,1.8828,4.6171,39.0000
2,primary_realistic,recent_node_median_20d,3.8446,2.6329,4.5547,39.0000
0,primary_realistic,global_median,4.4673,3.2421,5.6385,39.0000
7,stress_test,same_date_linear_interp,1.3975,1.2728,1.5903,39.0000
5,stress_test,node_median,3.6124,1.6997,4.7309,39.0000
6,stress_test,recent_node_median_20d,3.8365,2.5691,4.5259,39.0000
4,stress_test,global_median,5.0479,3.6372,6.3833,39.0000


In [9]:
demo_interp = predict_same_date_linear_interp(
    make_fold_bundle_for_protocol(train, fold_plan.iloc[0], "primary_realistic")["train_history"],
    make_fold_bundle_for_protocol(train, fold_plan.iloc[0], "primary_realistic")["val_masked"],
    lookback_dates=20,
)

print("Prediction source mix on Fold 1 / primary_realistic")
display(
    demo_interp.loc[demo_interp["is_scored_target"], "pred_source"]
    .value_counts(dropna=False)
    .rename_axis("pred_source")
    .to_frame("count")
)


Prediction source mix on Fold 1 / primary_realistic


,count
pred_source,
same_date_linear_interp,39


In [10]:
def add_eval_buckets(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["moneyness_bucket"] = pd.cut(
        out["moneyness"],
        bins=[-np.inf, 0.9, 1.1, np.inf],
        labels=["left_wing", "center", "right_wing"],
    )

    out["support_bucket"] = np.where(
        out["test_node_missing_rate"] >= 0.55,
        "harder_pattern",
        "easier_pattern",
    )

    return out


def breakdown_table(scored_df: pd.DataFrame, pred_col: str = "iv_pred", group_col: str = "maturity_label") -> pd.DataFrame:
    eval_df = scored_df.loc[scored_df["is_scored_target"]].copy()
    eval_df = eval_df.loc[eval_df["iv_observed"].notna() & eval_df[pred_col].notna()].copy()
    eval_df = add_eval_buckets(eval_df)

    return (
        eval_df.groupby(group_col)
        .apply(lambda g: pd.Series({"n": len(g), "rmse": rmse(g["iv_observed"], g[pred_col])}))
        .reset_index()
        .sort_values(group_col)
    )


In [11]:
def predict_same_date_nearest_anchor(
    train_history: pd.DataFrame,
    val_masked: pd.DataFrame,
    lookback_dates: int = 20,
) -> pd.DataFrame:
    out = predict_recent_node_median(train_history, val_masked, lookback_dates=lookback_dates).copy()

    out["nearest_anchor_pred"] = np.nan
    out["nearest_anchor_distance"] = np.nan
    out["nearest_anchor_source"] = pd.Series(index=out.index, dtype="object")

    for (_, maturity_label, option_type), g_idx in out.groupby(
        ["date", "maturity_label", "option_type"], sort=False
    ).groups.items():
        g = out.loc[g_idx].copy()

        anchors = (
            g.loc[g["is_visible_anchor"], ["moneyness", "iv_observed"]]
            .dropna()
            .sort_values("moneyness")
        )

        if len(anchors) == 0:
            continue

        anchor_x = anchors["moneyness"].to_numpy()
        anchor_y = anchors["iv_observed"].to_numpy()
        target_x = g["moneyness"].to_numpy()

        nearest_idx = np.abs(target_x[:, None] - anchor_x[None, :]).argmin(axis=1)
        pred_vals = anchor_y[nearest_idx]
        pred_dist = np.abs(target_x - anchor_x[nearest_idx])

        out.loc[g.index, "nearest_anchor_pred"] = pred_vals
        out.loc[g.index, "nearest_anchor_distance"] = pred_dist
        out.loc[g.index, "nearest_anchor_source"] = "same_date_nearest_anchor"

    use_nearest = out["nearest_anchor_pred"].notna()
    out.loc[use_nearest, "iv_pred"] = out.loc[use_nearest, "nearest_anchor_pred"]
    out["pred_source"] = np.where(use_nearest, out["nearest_anchor_source"], out["pred_source"])

    return out


In [12]:
nearest_results = pd.concat(
    [
        evaluate_baseline("primary_realistic", lambda th, vm: predict_same_date_nearest_anchor(th, vm, lookback_dates=20), "same_date_nearest_anchor"),
        evaluate_baseline("stress_test", lambda th, vm: predict_same_date_nearest_anchor(th, vm, lookback_dates=20), "same_date_nearest_anchor"),
    ],
    ignore_index=True,
)

results_phase3b = pd.concat([results_phase3, nearest_results], ignore_index=True)

summary_phase3b = (
    results_phase3b.groupby(["protocol", "baseline"])
    .agg(
        mean_rmse=("rmse", "mean"),
        min_rmse=("rmse", "min"),
        max_rmse=("rmse", "max"),
        mean_n_scored=("n_scored", "mean"),
    )
    .reset_index()
    .sort_values(["protocol", "mean_rmse"])
)

print("Baseline summary with nearest-anchor comparison")
display(summary_phase3b)


Baseline summary with nearest-anchor comparison


,protocol,baseline,mean_rmse,min_rmse,max_rmse,mean_n_scored
3,primary_realistic,same_date_linear_interp,1.1877,0.9115,1.5942,39.0000
4,primary_realistic,same_date_nearest_anchor,1.3662,1.1171,1.6774,39.0000
1,primary_realistic,node_median,3.6209,1.8828,4.6171,39.0000
2,primary_realistic,recent_node_median_20d,3.8446,2.6329,4.5547,39.0000
0,primary_realistic,global_median,4.4673,3.2421,5.6385,39.0000
8,stress_test,same_date_linear_interp,1.3975,1.2728,1.5903,39.0000
9,stress_test,same_date_nearest_anchor,1.4960,1.3616,1.5903,39.0000
6,stress_test,node_median,3.6124,1.6997,4.7309,39.0000
7,stress_test,recent_node_median_20d,3.8365,2.5691,4.5259,39.0000
5,stress_test,global_median,5.0479,3.6372,6.3833,39.0000


In [13]:
demo_linear = predict_same_date_linear_interp(
    make_fold_bundle_for_protocol(train, fold_plan.iloc[0], "primary_realistic")["train_history"],
    make_fold_bundle_for_protocol(train, fold_plan.iloc[0], "primary_realistic")["val_masked"],
    lookback_dates=20,
)

print("same_date_linear_interp breakdown by maturity_label")
display(breakdown_table(demo_linear, group_col="maturity_label"))

print("same_date_linear_interp breakdown by moneyness_bucket")
display(breakdown_table(demo_linear, group_col="moneyness_bucket"))

print("same_date_linear_interp breakdown by support_bucket")
display(breakdown_table(demo_linear, group_col="support_bucket"))


same_date_linear_interp breakdown by maturity_label


/var/folders/2r/65b3v1496498f35zwrwmt7900000gn/T/ipykernel_77238/3772523024.py:26: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({"n": len(g), "rmse": rmse(g["iv_observed"], g[pred_col])}))


,maturity_label,n,rmse
0,1M,10.0000,1.2537
1,2M,10.0000,2.6259
2,3M,10.0000,0.9600
3,6M,9.0000,0.7621


same_date_linear_interp breakdown by moneyness_bucket


/var/folders/2r/65b3v1496498f35zwrwmt7900000gn/T/ipykernel_77238/3772523024.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  eval_df.groupby(group_col)
/var/folders/2r/65b3v1496498f35zwrwmt7900000gn/T/ipykernel_77238/3772523024.py:26: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({"n": len(g), "rmse": rmse(g["iv_observed"], g[pred_col])}))


,moneyness_bucket,n,rmse
0,left_wing,13.0000,2.5745
1,center,13.0000,0.7775
2,right_wing,13.0000,0.6259


same_date_linear_interp breakdown by support_bucket


/var/folders/2r/65b3v1496498f35zwrwmt7900000gn/T/ipykernel_77238/3772523024.py:26: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({"n": len(g), "rmse": rmse(g["iv_observed"], g[pred_col])}))


,support_bucket,n,rmse
0,easier_pattern,28.0000,1.2536
1,harder_pattern,11.0000,2.2383


In [14]:
# --- Phase 3 polish / Cell 1: warning-free true-support diagnostics ---

def attach_true_local_support(scored_df: pd.DataFrame) -> pd.DataFrame:
    out = scored_df.copy()

    scored_targets = out.loc[out["is_scored_target"]].copy()
    visible_anchors = out.loc[out["is_visible_anchor"]].copy()

    support = local_support_profile(scored_targets, visible_anchors)[
        [
            "row_id",
            "opp_option_visible",
            "same_maturity_adj_visible_count",
            "same_moneyness_adj_visible_count",
        ]
    ].copy()

    support["any_local_same_date_support"] = (
        support["opp_option_visible"]
        | (support["same_maturity_adj_visible_count"] > 0)
        | (support["same_moneyness_adj_visible_count"] > 0)
    )
    support["hard_case"] = ~support["any_local_same_date_support"]

    out = out.merge(support, on="row_id", how="left")

    out["opp_option_visible"] = out["opp_option_visible"].fillna(False).astype(bool)
    out["same_maturity_adj_visible_count"] = out["same_maturity_adj_visible_count"].fillna(0).astype(int)
    out["same_moneyness_adj_visible_count"] = out["same_moneyness_adj_visible_count"].fillna(0).astype(int)
    out["any_local_same_date_support"] = out["any_local_same_date_support"].fillna(False).astype(bool)
    out["hard_case"] = out["hard_case"].fillna(False).astype(bool)

    out["wing_center_bucket"] = np.where(
        out["moneyness"].between(0.9, 1.1, inclusive="both"),
        "center",
        "wing",
    )
    out["hard_case_bucket"] = np.where(out["hard_case"], "hard_case", "non_hard_case")

    return out


def rmse_breakdown(df: pd.DataFrame, group_cols) -> pd.DataFrame:
    if isinstance(group_cols, str):
        group_cols = [group_cols]

    rows = []
    for keys, g in df.groupby(group_cols, observed=True, sort=True):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = {col: key for col, key in zip(group_cols, keys)}
        row["n"] = len(g)
        row["rmse"] = rmse(g["iv_observed"], g["iv_pred"])
        rows.append(row)

    return pd.DataFrame(rows).sort_values(group_cols).reset_index(drop=True)


In [15]:
# --- Phase 3 polish / Cell 2: pooled comparison for the two strongest baselines ---

def pooled_scored_predictions(protocol_name: str, baseline_name: str, predictor_fn) -> pd.DataFrame:
    pooled = []

    for row in fold_plan.itertuples(index=False):
        bundle = make_fold_bundle_for_protocol(train, row, protocol_name)
        scored = predictor_fn(bundle["train_history"], bundle["val_masked"]).copy()
        scored["fold"] = row.fold
        scored["protocol"] = protocol_name
        scored["baseline"] = baseline_name
        scored = attach_true_local_support(scored)
        pooled.append(scored.loc[scored["is_scored_target"]].copy())

    return pd.concat(pooled, ignore_index=True)


pooled_linear_primary = pooled_scored_predictions(
    "primary_realistic", "same_date_linear_interp", predict_same_date_linear_interp
)
pooled_linear_stress = pooled_scored_predictions(
    "stress_test", "same_date_linear_interp", predict_same_date_linear_interp
)
pooled_nearest_primary = pooled_scored_predictions(
    "primary_realistic", "same_date_nearest_anchor", predict_same_date_nearest_anchor
)
pooled_nearest_stress = pooled_scored_predictions(
    "stress_test", "same_date_nearest_anchor", predict_same_date_nearest_anchor
)

pooled_compare = pd.concat(
    [
        pooled_linear_primary,
        pooled_linear_stress,
        pooled_nearest_primary,
        pooled_nearest_stress,
    ],
    ignore_index=True,
)

print("Pooled overall comparison")
display(rmse_breakdown(pooled_compare, ["protocol", "baseline"]))

print("Pooled breakdown by maturity")
display(rmse_breakdown(pooled_compare, ["protocol", "baseline", "maturity_label"]))

print("Pooled breakdown by wing_center_bucket")
display(rmse_breakdown(pooled_compare, ["protocol", "baseline", "wing_center_bucket"]))

print("Pooled breakdown by hard_case_bucket")
display(rmse_breakdown(pooled_compare, ["protocol", "baseline", "hard_case_bucket"]))


/var/folders/2r/65b3v1496498f35zwrwmt7900000gn/T/ipykernel_77238/2655188324.py:27: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["opp_option_visible"] = out["opp_option_visible"].fillna(False).astype(bool)
/var/folders/2r/65b3v1496498f35zwrwmt7900000gn/T/ipykernel_77238/2655188324.py:30: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["any_local_same_date_support"] = out["any_local_same_date_support"].fillna(False).astype(bool)
/var/folders/2r/65b3v1496498f35zwrwmt7900000gn/T/ipykernel_77238/2655188324.py:31: FutureWarning: Downcasting object dtype

Pooled overall comparison


/var/folders/2r/65b3v1496498f35zwrwmt7900000gn/T/ipykernel_77238/2655188324.py:27: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["opp_option_visible"] = out["opp_option_visible"].fillna(False).astype(bool)
/var/folders/2r/65b3v1496498f35zwrwmt7900000gn/T/ipykernel_77238/2655188324.py:30: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["any_local_same_date_support"] = out["any_local_same_date_support"].fillna(False).astype(bool)
/var/folders/2r/65b3v1496498f35zwrwmt7900000gn/T/ipykernel_77238/2655188324.py:31: FutureWarning: Downcasting object dtype

,protocol,baseline,n,rmse
0,primary_realistic,same_date_linear_interp,156,1.2159
1,primary_realistic,same_date_nearest_anchor,156,1.3810
2,stress_test,same_date_linear_interp,156,1.4019
3,stress_test,same_date_nearest_anchor,156,1.4975


Pooled breakdown by maturity


,protocol,baseline,maturity_label,n,rmse
0,primary_realistic,same_date_linear_interp,1M,40,1.1354
1,primary_realistic,same_date_linear_interp,2M,39,1.6705
2,primary_realistic,same_date_linear_interp,3M,39,0.9625
3,primary_realistic,same_date_linear_interp,6M,38,0.9471
4,primary_realistic,same_date_nearest_anchor,1M,40,1.4983
5,primary_realistic,same_date_nearest_anchor,2M,39,1.8204
6,primary_realistic,same_date_nearest_anchor,3M,39,0.9519
7,primary_realistic,same_date_nearest_anchor,6M,38,1.0657
8,stress_test,same_date_linear_interp,1M,40,1.3384
9,stress_test,same_date_linear_interp,2M,39,2.0838


Pooled breakdown by wing_center_bucket


,protocol,baseline,wing_center_bucket,n,rmse
0,primary_realistic,same_date_linear_interp,center,65,0.7131
1,primary_realistic,same_date_linear_interp,wing,91,1.4734
2,primary_realistic,same_date_nearest_anchor,center,65,0.9748
3,primary_realistic,same_date_nearest_anchor,wing,91,1.6096
4,stress_test,same_date_linear_interp,center,49,0.7483
5,stress_test,same_date_linear_interp,wing,107,1.6153
6,stress_test,same_date_nearest_anchor,center,49,0.9213
7,stress_test,same_date_nearest_anchor,wing,107,1.6972


Pooled breakdown by hard_case_bucket


,protocol,baseline,hard_case_bucket,n,rmse
0,primary_realistic,same_date_linear_interp,hard_case,6,2.3840
1,primary_realistic,same_date_linear_interp,non_hard_case,150,1.1446
2,primary_realistic,same_date_nearest_anchor,hard_case,6,2.3840
3,primary_realistic,same_date_nearest_anchor,non_hard_case,150,1.3252
4,stress_test,same_date_linear_interp,hard_case,12,1.4597
5,stress_test,same_date_linear_interp,non_hard_case,144,1.3970
6,stress_test,same_date_nearest_anchor,hard_case,12,1.4783
7,stress_test,same_date_nearest_anchor,non_hard_case,144,1.4990


## Phase 3 baseline summary: methods, results, and insights

This notebook establishes the first serious benchmark layer for the IV-surface completion task.

The purpose of Phase 3 was not to build the final model.  
The goal was to answer a more basic question:

**How much of the problem can already be solved using simple, transparent, and reproducible baseline rules?**

All baselines were evaluated under both locked validation protocols from Phase 2:

- **`primary_realistic`**: optimistic / easier validation protocol
- **`stress_test`**: conservative / harder validation protocol

The actual competition test difficulty likely lies somewhere between these two settings, so baseline interpretation should rely on both rather than only one.

---

## 1. What we tried

We evaluated a sequence of baselines from weakest to strongest.

### `global_median`
Predict every target row using one global historical median IV.

Purpose:
- absolute floor benchmark
- no surface structure
- no node structure
- no same-date anchor use

### `node_median`
Predict each row using the historical median IV at the same:

- `maturity_label`
- `moneyness`
- `option_type`

Purpose:
- tests whether the fixed lattice itself contains strong reusable node-level information

### `recent_node_median_20d`
Use the node median computed only over the most recent 20 training dates, with fallback to full-history node medians.

Purpose:
- tests whether recent history is more useful than a larger historical sample

### `same_date_nearest_anchor`
Predict each target using the nearest visible same-date anchor in moneyness within the same:

- date
- maturity
- option type

Fallback:
- recent/full node medians

Purpose:
- tests whether simple same-date anchor copying is already strong

### `same_date_linear_interp`
Predict each target by linearly interpolating across visible same-date anchors in moneyness within the same:

- date
- maturity
- option type

Fallback:
- recent/full node medians

Purpose:
- tests whether the local same-date smile shape can already be captured with a simple interpolation rule

---

## 2. Main baseline ranking

The baseline ordering is stable under both validation protocols:

1. `same_date_linear_interp`
2. `same_date_nearest_anchor`
3. `node_median`
4. `recent_node_median_20d`
5. `global_median`

This ranking is one of the most important outputs of the notebook.

It means:

- same-date anchor-aware baselines are dramatically stronger than historical-only baselines
- interpolation between anchors is better than copying only the nearest anchor
- simple recency restriction does not improve node medians in this form
- the baseline hierarchy is robust across both optimistic and conservative validation settings

---

## 3. Key findings

### A. Same-date anchors dominate historical-only information

The strongest result of Phase 3 is that same-date anchor-aware baselines outperform historical-only baselines by a large margin.

Interpretation:
- the competition is fundamentally a **same-date partial surface completion** problem
- historical node averages help, but they are not the main signal
- visible same-date anchors contain the most important predictive information

This is fully consistent with the Phase 1 EDA and Phase 2 validation analysis.

### B. The fixed lattice still matters

Among the historical-only baselines:

- `node_median` clearly beats `global_median`
- `recent_node_median_20d` does not beat `node_median`

Interpretation:
- node identity matters a lot
- but reducing the sample to only recent dates loses too much information in this simple median setup
- at this stage, larger historical support is more useful than recency-only medians

### C. Interpolation matters beyond nearest-anchor copying

`same_date_linear_interp` consistently beats `same_date_nearest_anchor` in pooled results across all folds and under both protocols.

Interpretation:
- it is not enough to copy the closest visible anchor
- the shape between visible anchors contains real information
- even a simple linear smile interpolation rule captures useful structure that nearest-anchor copying misses

This is the clearest sign that the next modeling phase should focus on **structured surface fitting**, not just smarter lookup heuristics.

---

## 4. What the pooled diagnostics show

To avoid over-interpreting one demo fold, we also compared the two strongest baselines using pooled scored targets across all folds.

### Pooled overall comparison
The same conclusion holds at the pooled level:

- `same_date_linear_interp` is better than `same_date_nearest_anchor`
- under both `primary_realistic` and `stress_test`

So the main baseline conclusion is not a one-fold artifact.

### By maturity
The maturity breakdown shows that difficulty is uneven across the term structure.

In the pooled results:
- `2M` appears to be the hardest maturity
- `3M` and `6M` are easier
- `1M` is still nontrivial, but not always the worst slice

Interpretation:
- structured models should be maturity-aware
- diagnostics should not assume only one maturity matters
- the short end is important, but difficulty is not confined only to `1M`

### By wing vs center
This is one of the clearest patterns in the notebook.

The center of the moneyness range is much easier than the wings.

Interpretation:
- wing regions remain materially harder
- local interpolation helps, but does not fully solve wing difficulty
- cross-moneyness smoothness is one of the main remaining modeling opportunities

### By hard-case vs non-hard-case
Using true local support diagnostics, hard-case rows are harder than non-hard-case rows, especially under the easier protocol.

Interpretation:
- local same-date support continues to be a major driver of difficulty
- interpolation helps most when enough same-date geometry exists
- once support becomes weak enough, interpolation and nearest-anchor methods converge toward fallback-like behavior

This gives a concrete target for later models:
- improve performance in low-support and locally sparse regions

---

## 5. What Phase 3 tells us about the problem

Phase 3 gives a strong answer to a core project question:

**Can a simple same-date completion rule already solve a large share of the task?**

Yes.

A basic same-date linear interpolation baseline already performs very well relative to all historical-only alternatives.

This means:

- the same-date anchor structure is highly informative
- local cross-moneyness shape matters
- any future model must beat a strong anchor-aware interpolation benchmark, not just naive historical averages

At the same time, the notebook also tells us exactly where the remaining gap is concentrated:

- wings
- harder-support / hard-case rows
- some maturity-specific difficult slices
- regions where a simple piecewise linear rule is too crude

These are exactly the areas that Phase 4 should target.

---

## 6. Final baseline conclusion

Phase 3 establishes the following benchmark picture:

### Weak floor benchmarks
- `global_median`
- `node_median`
- `recent_node_median_20d`

These are useful as references, but they are not competitive with same-date anchor-aware methods.

### Strong benchmark baselines
- `same_date_nearest_anchor`
- `same_date_linear_interp`

These are the first genuinely competitive baseline families.

Among them:

- `same_date_linear_interp` is the strongest Phase 3 benchmark
- its advantage over nearest-anchor copying is real
- the ranking is stable across both validation protocols
- the pooled diagnostics confirm the result

### What this means for Phase 4
The next step should not be generic machine learning yet.

The next logical step is to build **structured surface models** that improve on simple linear interpolation by imposing smoother, more finance-aware structure across:

- moneyness
- maturity
- and, where useful, total variance

So the output of Phase 3 is:

- a validated baseline hierarchy
- a strong benchmark to beat
- pooled evidence about where the current benchmark succeeds and fails
- and a clear direction for the next notebook

The project can now move into **Phase 4: structured surface models**.
